# Entrenar en Kaggle (2×T4) — issue #4

SFT con LoRA de Gemma 4 E4B sobre `datos/conversaciones/` con el prompt de la cátedra. Misma infraestructura que `correr.ipynb`
(clona la rama, instala las versiones fijas, reparte el modelo entre las dos GPU), más W&B y el guardado de checkpoints.

Antes: *Settings* → Accelerator **GPU T4 x2**, Internet **on**, y en *Add-ons* → *Secrets*:
- `HF_TOKEN`: token de Hugging Face con **escritura** sobre el repo privado del grupo (los checkpoints se copian ahí para poder reanudar).
- `WANDB_API_KEY`: clave de W&B.

Orden recomendado: 1) `ENSAYO = True` (unos minutos: comprueba memoria y el flujo), 2) corrida completa con *Save & Run All*,
3) si la sesión se corta por el límite, `REANUDAR = True` con el mismo `CONFIG`. Detalle en `entrenar/README.md`.

In [ ]:
# Parámetros: lo único que se cambia entre corridas.
RAMA = "juli/entrenamiento"
COMMIT = ""            # vacío: la punta de la rama; o un hash para fijarlo
CONFIG = "entrenar/configs/sft-catedra.json"
ENSAYO = True          # True: corrida corta (no se reporta). False: la corrida real.
REANUDAR = False       # True: seguir desde el último checkpoint (local o el copiado en Hugging Face)
WANDB_ENTIDAD = ""     # vacío: el del config (gen-ai-grupo-2)
WANDB_BASE_URL = "https://forge.coreweave.com/wandb"  # el W&B del grupo no es wandb.ai; ver entrenar/README.md
LOTE = ""              # no se usa al entrenar; queda por compatibilidad con correr_kaggle.py

In [ ]:
import os, subprocess
from kaggle_secrets import UserSecretsClient

secretos = UserSecretsClient()
os.environ["HF_TOKEN"] = secretos.get_secret("HF_TOKEN")
os.environ["WANDB_API_KEY"] = secretos.get_secret("WANDB_API_KEY")
if WANDB_ENTIDAD:
    os.environ["WANDB_ENTITY"] = WANDB_ENTIDAD
if WANDB_BASE_URL:
    os.environ["WANDB_BASE_URL"] = WANDB_BASE_URL
os.environ["PYTHONUTF8"] = "1"
REPO = "/tmp/repo"  # fuera de /kaggle/working, para que el .venv no cuente como salida

def sh(comando):
    print("$", comando, flush=True)
    subprocess.run(comando, shell=True, check=True, cwd=REPO if os.path.isdir(REPO) else None)

sh("nvidia-smi")
if not os.path.isdir(REPO):
    sh(f"git clone -q -b {RAMA} https://github.com/genai-grupo-2/emergency-admission-agent.git {REPO}")
else:
    sh(f"git fetch -q origin {RAMA} && git checkout -q {RAMA} && git reset -q --hard origin/{RAMA}")
if COMMIT:
    sh(f"git checkout -q {COMMIT}")
sh("git log --oneline -1 && git status --short")

In [ ]:
# Mismas versiones que la imagen de modal_app.py, instaladas en el .venv del repo (--python: Kaggle configura uv
# para instalar en el Python del sistema). Si la rueda de torch por defecto no ve las GPU (driver viejo), se prueba
# la que elige uv según el driver y después las de otras versiones de CUDA. La que quedó se anota en corridas/entornos/.
sh("pip install -q uv")
sh("uv sync --locked -q")
PIP = "uv pip install -q --python .venv/bin/python"
sh(f"{PIP} torch==2.14.1 transformers==5.20.2 accelerate==1.15.0 peft==0.21.3")
ver = (".venv/bin/python -c \"import torch; print(torch.__version__, torch.version.cuda, torch.cuda.is_available(), "
       "torch.cuda.device_count()); assert torch.cuda.device_count() >= 2\"")
intentos = [None, "--torch-backend=auto"] + [f"--index-url https://download.pytorch.org/whl/{c}"
                                           for c in ["cu130", "cu129", "cu128", "cu126"]]
for intento in intentos:
    if intento and subprocess.run(f"{PIP} --reinstall torch==2.14.1 {intento}", shell=True, cwd=REPO).returncode:
        print("no hay torch 2.14.1 con", intento)
        continue
    if subprocess.run(ver, shell=True, cwd=REPO).returncode == 0:
        print("torch listo con", intento or "la rueda por defecto de PyPI")
        break
else:
    raise RuntimeError("torch 2.14.1 no ve las dos GPU con ninguna rueda: pasar la salida de nvidia-smi y de esta celda")

In [ ]:
# Falla rápido, antes de cargar el modelo, si la clave o el servidor de W&B están mal (ver WANDB_BASE_URL arriba).
sh(".venv/bin/python -c \"import wandb; assert wandb.login(verify=True), 'W&B rechazó la clave o el servidor'; print('W&B OK')\"")

In [ ]:
import json
nombre = json.load(open(f"{REPO}/{CONFIG}", encoding="utf-8"))["nombre"] + ("-ensayo" if ENSAYO else "")
CHECKPOINTS = f"/kaggle/working/checkpoints/{nombre}"   # fuera de corridas/: pesan, no van al zip ni a Git
sh(".venv/bin/python -m entrenar.entrenar --config " + CONFIG + " --solo-datos")  # valida datos y prompt, sin GPU

In [ ]:
argumentos = f"--config {CONFIG} --checkpoints {CHECKPOINTS}"
argumentos += " --ensayo" if ENSAYO else ""
argumentos += " --reanudar" if REANUDAR else ""
# correr_kaggle.py anota commit, GPU y versiones en corridas/entornos/ y deja el zip de corridas/ para bajar.
sh(f".venv/bin/python correr_kaggle.py --comando \"entrenar.entrenar {argumentos}\"")

## Después de entrenar

1. Bajar `corridas-kaggle-<fecha>.zip` del panel *Output*, descomprimirlo en la raíz del repo y commitear
   `corridas/entrenamiento/<nombre>/` y `corridas/entornos/kaggle-<fecha>.json` (el manifiesto es lo que audita la corrida).
2. Publicar el checkpoint elegido por la pérdida de validación (en otra celda o sesión, con la misma GPU):
   `python -m entrenar.publicar --nombre <nombre> --paso <N> --repo <usuario-o-org>/<repo> --checkpoints /kaggle/working/checkpoints/<nombre>`
3. Verificar la carga desde cero: `python -m entrenar.verificar_carga --repo <repo> --revision <hash>`.